# GBasis libcint Tutorial

This notebook demonstrates how to use the `CBasis` class from `gbasis.integrals.libcint` to compute GTO integrals using C shell-loop implementations backed by the [libcint](https://github.com/sunqm/libcint) library.

**What you will learn:**
- How to set up a `CBasis` instance for a molecule
- How to compute 1-electron integrals (overlap, kinetic, nuclear attraction)
- How to compute 2-electron repulsion integrals (ERI)
- How to compute gradient integrals (building blocks for nuclear gradients)
- How to compute GIAO/magnetic integrals (building blocks for NMR properties)
- How to compute 3-center 2-electron integrals (used in density fitting)

**Prerequisites:**
- GBasis installed with libcint support (`pip install gbasis`)
- A basis set file in NWChem format (e.g. `sto-3g.nwchem`)


## 1. Setup

Import the necessary modules and define the molecule. Atomic coordinates must be in **Bohr** (atomic units).

In [1]:
import numpy as np
import numpy.testing as npt

from gbasis.parsers import make_contractions, parse_nwchem
from gbasis.integrals.libcint import CBasis

In [2]:
# Define a water molecule in Bohr (atomic units)
atsyms = ["O", "H", "H"]
atcoords = np.array([
    [ 0.000000,  0.000000,  0.000000],  # O
    [ 0.000000,  1.430429,  1.107157],  # H
    [ 0.000000, -1.430429,  1.107157],  # H
])

print(f"Molecule: {atsyms}")
print(f"Coordinates (Bohr):\n{atcoords}")

Molecule: ['O', 'H', 'H']
Coordinates (Bohr):
[[ 0.        0.        0.      ]
 [ 0.        1.430429  1.107157]
 [ 0.       -1.430429  1.107157]]


In [3]:
# Parse the STO-3G basis set and build contractions
# Replace 'sto-3g.nwchem' with the path to your basis set file
# GBasis test data: tests/data/data_sto6g.nwchem

basis_dict = parse_nwchem("/Users/aayushgupta/Desktop/GSOC26/Gbasis/gbasis/tests/data/data_sto6g.nwchem")py_basis = make_contractions(basis_dict, atsyms, atcoords, coord_types="spherical")

# Build the CBasis object (spherical coordinates)
cb = CBasis(py_basis, atsyms, atcoords, coord_type="spherical")

print(f"Number of atoms:           {cb.natm}")
print(f"Number of shells:          {cb.nbas}")
print(f"Number of basis functions: {cb.nbfn}")
print(f"Coordinate type:           {cb.coord_type}")

FileNotFoundError: [Errno 2] No such file or directory: 'tests/data/data_sto6g.nwchem'

## 2. One-Electron Integrals

The following 1-electron integrals are available via `CBasis`:

| Method | Integral | Description |
|--------|----------|-------------|
| `cb.overlap()` | $S_{ij} = \langle \phi_i \| \phi_j \rangle$ | Overlap matrix |
| `cb.kinetic_energy()` | $T_{ij} = \langle \phi_i \| -\frac{1}{2}\nabla^2 \| \phi_j \rangle$ | Kinetic energy |
| `cb.nuclear_attraction()` | $V_{ij} = \langle \phi_i \| \sum_A Z_A/r_A \| \phi_j \rangle$ | Nuclear attraction |
| `cb.momentum()` | $p_{ij} = \langle \phi_i \| -i\nabla \| \phi_j \rangle$ | Momentum (complex) |
| `cb.rinv()` | $V_{ij} = \langle \phi_i \| 1/r \| \phi_j \rangle$ | 1/r operator |

In [ ]:
# Overlap integral
S = cb.overlap()
print(f"Overlap matrix shape: {S.shape}")
print(f"Overlap diagonal (should be ~1.0 for normalized basis):")
print(np.diag(S).round(6))

# Verify symmetry
npt.assert_allclose(S, S.T, atol=1e-12)
print("Overlap matrix is symmetric: OK")

In [ ]:
# Kinetic energy integral
T = cb.kinetic_energy()
print(f"Kinetic energy matrix shape: {T.shape}")
print(f"Kinetic energy diagonal (all positive):")
print(np.diag(T).round(6))
assert np.all(np.diag(T) > 0), "Kinetic energy diagonal should be positive"
print("Kinetic energy diagonal is positive: OK")

In [ ]:
# Nuclear attraction integral
V = cb.nuclear_attraction()
print(f"Nuclear attraction matrix shape: {V.shape}")
print(f"Nuclear attraction diagonal (all negative):")
print(np.diag(V).round(6))

In [ ]:
# Core Hamiltonian H = T + V
H_core = T + V
print(f"Core Hamiltonian diagonal:")
print(np.diag(H_core).round(6))

In [ ]:
# Momentum integral (purely imaginary: p = -i * real_buffer)
p = cb.momentum(origin=np.zeros(3))
print(f"Momentum integral shape: {p.shape}  (nbfn x nbfn x 3 components)")
print(f"Momentum is complex: {np.iscomplexobj(p)}")

# Momentum is anti-Hermitian: p_ij = -p_ji*
npt.assert_allclose(p[:, :, 0], -p[:, :, 0].conj().T, atol=1e-10)
print("Momentum is anti-Hermitian: OK")

### Optional: MO Transformation

All `CBasis` methods accept an optional `transform` matrix to convert from AO to MO basis.
For example, if `C` is the MO coefficient matrix (shape `[nbfn, nmo]`):

```python
S_mo = cb.overlap(transform=C.T)   # shape: [nmo, nmo]
T_mo = cb.kinetic_energy(transform=C.T)
```

## 3. Electron Repulsion Integrals (ERI)

The 2-electron repulsion integral is:

$$g_{ijkl} = \langle \phi_i \phi_j \| \frac{1}{r_{12}} \| \phi_k \phi_l \rangle$$

Two index conventions are supported:
- **Physicist notation** (default): `out[i, j, k, l]` = $\langle ij | kl \rangle$
- **Chemist notation**: `out[i, j, k, l]` = $(ij|kl)$

In [ ]:
# Electron repulsion integrals (physicist notation, default)
eri = cb.electron_repulsion(notation="physicist")
print(f"ERI shape: {eri.shape}  (nbfn x nbfn x nbfn x nbfn)")
print(f"ERI[0,0,0,0] = {eri[0,0,0,0]:.6f}  (should be positive)")
assert np.all(np.isfinite(eri)), "ERI contains NaN or Inf"
print("ERI is finite: OK")

In [ ]:
# Verify 8-fold symmetry: g_ijkl = g_jikl = g_ijlk = g_klij = ...
npt.assert_allclose(eri, eri.transpose(1, 0, 2, 3), atol=1e-10)  # i <-> j
npt.assert_allclose(eri, eri.transpose(0, 1, 3, 2), atol=1e-10)  # k <-> l
npt.assert_allclose(eri, eri.transpose(2, 3, 0, 1), atol=1e-10)  # ij <-> kl
print("ERI 8-fold symmetry: OK")

## 4. Moment Integrals

The `moment()` method computes multipole moment integrals up to 3rd order:

$$M_{ij} = \langle \phi_i | (x-X_0)^{n_x}(y-Y_0)^{n_y}(z-Z_0)^{n_z} | \phi_j \rangle$$

In [ ]:
origin = np.zeros(3)

# Compute overlap (order 0), dipole (order 1), and quadrupole (order 2) moments
orders = np.array([
    [0, 0, 0],  # overlap
    [1, 0, 0],  # x dipole
    [0, 1, 0],  # y dipole
    [0, 0, 1],  # z dipole
    [2, 0, 0],  # xx quadrupole
    [0, 2, 0],  # yy quadrupole
    [0, 0, 2],  # zz quadrupole
])

M = cb.moment(orders, origin=origin)
print(f"Moment integral shape: {M.shape}  (nbfn x nbfn x n_orders)")

# Order 0 should equal the overlap matrix
npt.assert_allclose(M[:, :, 0], S, atol=1e-10)
print("Moment order 0 == overlap matrix: OK")

## 5. Gradient Integrals

Gradient integrals are building blocks for computing nuclear coordinate gradients of the energy.

| Method | libcint function | Description |
|--------|-----------------|-------------|
| `cb.gradient_kinetic()` | `int1e_ipkin` | $i\nabla T$ |
| `cb.gradient_nuclear()` | `int1e_ipnuc` | $i\nabla V$ |
| `cb.gradient_rinv()` | `int1e_iprinv` | $i\nabla (1/r)$ |

In [ ]:
# Gradient of kinetic energy integral
ipkin = cb.gradient_kinetic()
print(f"Gradient kinetic shape: {ipkin.shape}")
assert np.all(np.isfinite(ipkin)), "gradient_kinetic contains NaN or Inf"
print(f"gradient_kinetic is finite: OK")

# Gradient of nuclear attraction integral
ipnuc = cb.gradient_nuclear()
print(f"Gradient nuclear shape: {ipnuc.shape}")
assert np.all(np.isfinite(ipnuc))
print(f"gradient_nuclear is finite: OK")

# Gradient of 1/r integral
iprinv = cb.gradient_rinv(inv_origin=np.zeros(3))
print(f"Gradient rinv shape: {iprinv.shape}")
assert np.all(np.isfinite(iprinv))
print(f"gradient_rinv is finite: OK")

## 6. GIAO / Magnetic Integrals

Gauge-including atomic orbital (GIAO) integrals are building blocks for NMR shielding tensors and magnetic susceptibilities.

| Method | libcint function | Description |
|--------|-----------------|-------------|
| `cb.ia01p()` | `int1e_ia01p` | GIAO paramagnetic shielding |
| `cb.ircxp()` | `int1e_cg_irxp` | GIAO angular momentum |
| `cb.iking()` | `int1e_igkin` | GIAO kinetic energy |
| `cb.iovlpg()` | `int1e_igovlp` | GIAO overlap gradient |
| `cb.inucg()` | `int1e_ignuc` | GIAO nuclear attraction |

In [ ]:
# GIAO paramagnetic shielding
ia01p = cb.ia01p()
print(f"ia01p shape: {ia01p.shape}")
assert np.all(np.isfinite(ia01p))
print("ia01p is finite: OK")

# GIAO angular momentum
ircxp = cb.ircxp()
print(f"ircxp shape: {ircxp.shape}")
assert np.all(np.isfinite(ircxp))
print("ircxp is finite: OK")

# GIAO kinetic energy
iking = cb.iking()
print(f"iking shape: {iking.shape}")
assert np.all(np.isfinite(iking))
print("iking is finite: OK")

# GIAO overlap gradient
iovlpg = cb.iovlpg()
print(f"iovlpg shape: {iovlpg.shape}")
assert np.all(np.isfinite(iovlpg))
print("iovlpg is finite: OK")

# GIAO nuclear attraction
inucg = cb.inucg()
print(f"inucg shape: {inucg.shape}")
assert np.all(np.isfinite(inucg))
print("inucg is finite: OK")

## 7. 3-Center 2-Electron Integrals

The 3-center 2-electron integral is used in density fitting (resolution of the identity) approximations:

$$(ij|k) = \langle \phi_i \phi_j \| \frac{1}{r_{12}} \| \phi_k \rangle$$

This integral exploits $i \leq j$ symmetry: `out[i, j, k] = out[j, i, k]`.

In [ ]:
# 3-center 2-electron integrals
int3c2e = cb.three_center_two_electron()
print(f"3c2e shape: {int3c2e.shape}  (nbfn x nbfn x nbfn)")
assert np.all(np.isfinite(int3c2e))
print("3c2e is finite: OK")

# Verify i <-> j symmetry
npt.assert_allclose(int3c2e, int3c2e.transpose(1, 0, 2), atol=1e-10)
print("3c2e i<->j symmetry: OK")

## 8. Cartesian Basis

`CBasis` also supports cartesian coordinates. Simply pass `coord_type="cartesian"`.

In [ ]:
# Build CBasis with cartesian coordinates
py_basis_cart = make_contractions(basis_dict, atsyms, atcoords, coord_types="cartesian")
cb_cart = CBasis(py_basis_cart, atsyms, atcoords, coord_type="cartesian")

print(f"Cartesian basis functions: {cb_cart.nbfn}")
print(f"Spherical basis functions: {cb.nbfn}")

# Overlap diagonal should be 1 for cartesian too
S_cart = cb_cart.overlap()
print(f"Cartesian overlap diagonal:")
print(np.diag(S_cart).round(6))

## 9. Verification Against GBasis Python Reference

Here we verify that the C shell-loop results match the pure-Python GBasis reference implementations.

In [ ]:
from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.nuclear_electron_attraction import nuclear_electron_attraction_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral_improved
from gbasis.integrals.libcint import ELEMENTS

atnums = np.array([ELEMENTS.index(s) for s in atsyms], dtype=float)
atol = 1e-6

# Overlap
py_S = overlap_integral(py_basis, screen_basis=False)
npt.assert_allclose(cb.overlap(), py_S, atol=atol)
print("Overlap matches Python reference: OK")

# Kinetic energy
py_T = kinetic_energy_integral(py_basis, screen_basis=False)
npt.assert_allclose(cb.kinetic_energy(), py_T, atol=atol)
print("Kinetic energy matches Python reference: OK")

# Nuclear attraction
py_V = nuclear_electron_attraction_integral(py_basis, atcoords, atnums)
npt.assert_allclose(cb.nuclear_attraction(), py_V, atol=atol)
print("Nuclear attraction matches Python reference: OK")

# ERI (looser tolerance — 4-index integral)
py_eri = electron_repulsion_integral_improved(py_basis)
npt.assert_allclose(cb.electron_repulsion(), py_eri, atol=1e-4, rtol=1e-5)
print("ERI matches Python reference: OK")

## Summary

| Integral | Method | Shape | Notes |
|----------|--------|-------|-------|
| Overlap | `cb.overlap()` | `(N, N)` | Symmetric |
| Kinetic energy | `cb.kinetic_energy()` | `(N, N)` | Symmetric, positive diagonal |
| Nuclear attraction | `cb.nuclear_attraction()` | `(N, N)` | Symmetric |
| 1/r | `cb.rinv()` | `(N, N)` | Symmetric |
| Momentum | `cb.momentum()` | `(N, N, 3)` | Complex, anti-Hermitian |
| Moment | `cb.moment(orders)` | `(N, N, M)` | M = number of orders |
| ERI | `cb.electron_repulsion()` | `(N, N, N, N)` | 8-fold symmetry |
| Gradient kinetic | `cb.gradient_kinetic()` | `(N, N)` | |
| Gradient nuclear | `cb.gradient_nuclear()` | `(N, N)` | |
| Gradient rinv | `cb.gradient_rinv()` | `(N, N)` | |
| GIAO ia01p | `cb.ia01p()` | `(N, N)` | NMR building block |
| GIAO ircxp | `cb.ircxp()` | `(N, N)` | NMR building block |
| GIAO iking | `cb.iking()` | `(N, N)` | NMR building block |
| GIAO iovlpg | `cb.iovlpg()` | `(N, N)` | NMR building block |
| GIAO inucg | `cb.inucg()` | `(N, N)` | NMR building block |
| 3c2e | `cb.three_center_two_electron()` | `(N, N, N)` | i<=j symmetry |

All methods accept an optional `transform` matrix to convert from AO to MO basis.